# Phase 1 Main Pipeline: Automated Sleep Staging & Wearable EEG Channel Simulation

This notebook implements and executes the complete **Phase 1 Pipeline** step by step on the 10 **SHHS1** Polysomnography (PSG) recordings in `data/raw/`:

1. **Step 1 — Data Ingestion & Standardized Preprocessing** (`src/data/preprocessing.py`)
   - Reads `.edf` physiological signals (`EEG` = `C4-A1`, `EEG(sec)` = `C3-A2`, `EOG(L)`, `EOG(R)`, `EMG`) and Compumedics Profusion `.xml` annotations.
   - Maps sleep stages to the 5 AASM classes (`0: Wake`, `1: N1`, `2: N2`, `3: N3 [merges R&K stages 3 & 4]`, `4: REM`).
   - Applies a 60 Hz notch filter, polyphase resampling to **100 Hz**, zero-phase **5th-order Butterworth bandpass filtering (0.2–49 Hz)**, and segmentation into **30-second epochs** (`3000` samples/epoch).
2. **Step 2 — Channel Ablation & Wearable Simulation** (`src/data/channel_sim.py`)
   - **Category A (Clinical Baselines):** `full_psg` (`5ch`), `eeg_eog` (`4ch`), `dual_eeg` (`2ch`).
   - **Category B (Option 1 — Location-Close Proxies):** `single_eeg_c4` (`C4-A1`), `single_eeg_c3` (`C3-A2`), `single_eog_l` (`EOG(L)` near `AF7/Fp1`), `dual_eog` (`EOG(L), EOG(R)`).
   - **Category C (Option 2 — Mathematical Wearable Derivations):**
     - `sim_bipolar_eog`: $\text{EOG(L)} - \text{EOG(R)}$ (Mastoid-free cross-forehead/temple strip `AF7-AF8` / `Fp1-Fp2`).
     - `sim_avg_eog`: $\frac{\text{EOG(L)} + \text{EOG(R)}}{2}$ (Mastoid-referenced midline forehead sensor `Fpz-M2`).
     - `sim_bipolar_eeg`: $(\text{C3-A2}) - (\text{C4-A1})$ (Cross-hemisphere bipolar subtraction, testing bilateral slow-wave cancellation).
     - `sim_avg_eeg`: $\frac{(\text{C3-A2}) + (\text{C4-A1})}{2}$ (Mastoid-referenced midline central sensor).
     - `sim_eeg_minus_eog`: $(\text{C4-A1}) - \text{EOG(L)}$ (Mastoid-free fronto-central differential).
3. **Step 3 — TinySleepNet Architecture** (`src/models/tinysleepnet.py`)
   - Modular PyTorch implementation supporting **Model A (`use_context=False`, Context-Blind CNN)** and **Model B (`use_context=True`, Context-Aware CNN + Unidirectional LSTM)** via a single boolean toggle.
4. **Step 4 — Subject-Independent 10-Fold Cross-Validation Trainer** (`src/training/trainer.py`)
   - Strict subject-level 80% Train / 10% Validation / 10% Test splits saved to `data/splits/shhs_10fold_splits.json`.
5. **Step 5 — Standardized Evaluation Framework** (`src/evaluation/metrics.py` & `src/evaluation/clinical_sleep.py`)
   - Exact **Markov et al. (2025)** Equations 1–29 (Multiclass $\kappa$, Multiclass MCC, Micro/Macro/Per-Stage ACC, SE, SP, PPV, NPV, F1), Normalized Confusion Matrices, Whole-Night Clinical Sleep Parameters (`TST`, `SE%`, `SOL`, `WASO`), Hypnograms, and **Bland–Altman** plots.

---
## Step 0: Install `requirements.txt` & Verify All Required Libraries

This first cell ensures all packages listed in `requirements.txt` (`mne`, `pyedflib`, `pypdf`, `pymupdf`, `torch`, `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`, `pyyaml`, `tqdm`, `h5py`) are installed in the active notebook kernel before running the pipeline.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

req_file = PROJECT_ROOT / "requirements.txt"
print(f"Installing dependencies from {req_file} into kernel: {sys.executable}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)])

REQUIRED_MODULES = [
    ("mne", "mne"),
    ("pyedflib", "pyedflib"),
    ("pypdf", "pypdf"),
    ("pymupdf", "fitz"),
    ("torch", "torch"),
    ("numpy", "numpy"),
    ("scipy", "scipy"),
    ("scikit-learn", "sklearn"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("pyyaml", "yaml"),
    ("tqdm", "tqdm"),
    ("h5py", "h5py"),
]

print("\nVerified Installed Libraries:")
for pkg_name, mod_name in REQUIRED_MODULES:
    mod = importlib.import_module(mod_name)
    ver = getattr(mod, "__version__", "installed")
    print(f"  - {pkg_name:15s} ({mod_name}): {ver}")

In [ ]:
import os
import sys
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocessing import (
    preprocess_all_recordings,
    load_preprocessed_subject,
    AASM_STAGE_NAMES,
)
from src.data.channel_sim import (
    load_ablation_configs,
    apply_channel_config,
)
from src.data.dataset import (
    SleepStageDataset,
    generate_subject_cv_splits,
    load_subject_cv_splits,
    load_subject_records_from_dir,
)
from src.models.tinysleepnet import TinySleepNet, build_tinysleepnet_from_config
from src.training.trainer import run_cross_validation_for_config, resolve_device
from src.evaluation.metrics import (
    compute_all_epoch_metrics,
    metrics_summary_to_dataframe,
    plot_normalized_confusion_matrix,
)
from src.evaluation.clinical_sleep import (
    compute_hypnogram_parameters,
    compute_bland_altman_agreement,
    plot_bland_altman_grid,
    plot_hypnogram_comparison,
)

with open(PROJECT_ROOT / "configs/base_config.yaml", "r", encoding="utf-8") as f:
    base_config = yaml.safe_load(f)

shhs_configs = load_ablation_configs(PROJECT_ROOT / "configs/ablation_channels.yaml", section="shhs_configs")
full_10_20_configs = load_ablation_configs(PROJECT_ROOT / "configs/ablation_channels.yaml", section="full_10_20_configs")

print(f"Project Root : {PROJECT_ROOT}")
print(f"PyTorch      : {torch.__version__} | Device: {resolve_device(base_config['training']['device'])}")
print(f"SHHS Configs : {len(shhs_configs)} loaded ({list(shhs_configs.keys())})")

---
## Step 1: Data Ingestion & Standardized Preprocessing (`src/data/preprocessing.py`)

We now preprocess all 10 SHHS1 EDF files (`shhs1-200001.edf` .. `shhs1-200010.edf`) and their corresponding Profusion XML annotations into standardized `100 Hz`, 30-second epoch `.npz` archives in `data/processed/`.

In [ ]:
saved_npz_paths = preprocess_all_recordings(base_config, root_dir=PROJECT_ROOT, overwrite=False)
all_records = load_subject_records_from_dir(PROJECT_ROOT / base_config["paths"]["processed_dir"])

summary_rows = []
for rec in all_records:
    labels = rec["labels"]
    row = {
        "Subject_ID": rec["subject_id"],
        "N_Epochs": len(labels),
        "Duration_Hours": round(len(labels) * 30.0 / 3600.0, 2),
        "Channels": ", ".join(rec["channel_names"]),
    }
    for idx, st_name in enumerate(AASM_STAGE_NAMES):
        row[f"{st_name}_epochs"] = int((labels == idx).sum())
    summary_rows.append(row)

df_subjects = pd.DataFrame(summary_rows)
display(df_subjects)
print(f"Total 30s epochs across {len(all_records)} subjects: {df_subjects['N_Epochs'].sum():,}")

---
## Step 2: Channel Ablation & Mathematical Wearable Simulation (`src/data/channel_sim.py`)

Let's inspect all 12 SHHS1 channel configurations and verify the shapes and biophysical properties of the simulated channels (especially comparing mastoid-referenced bilateral averaging vs. mastoid-free bipolar subtraction on an N3 Slow-Wave Sleep epoch).

In [ ]:
sample_subj = all_records[0]
config_inspection_rows = []

for cfg_name, cfg_spec in shhs_configs.items():
    derived_x, ch_names = apply_channel_config(sample_subj, cfg_spec, normalize=True)
    config_inspection_rows.append({
        "Config_Name": cfg_name,
        "Category": cfg_spec.get("category", ""),
        "Mastoid_Ref": cfg_spec.get("mastoid", False),
        "Out_Shape": str(derived_x.shape),
        "Derived_Channels": ", ".join(ch_names),
        "Description": cfg_spec.get("description", ""),
    })

df_configs = pd.DataFrame(config_inspection_rows)
display(df_configs)

In [ ]:
# Visualize a 10-second window of an N3 (Slow-Wave Sleep) epoch across Location-Close vs. Mathematically Simulated channels
n3_indices = np.where(sample_subj["labels"] == 3)[0]
ep_idx = int(n3_indices[len(n3_indices) // 2]) if len(n3_indices) > 0 else 0
t_sec = np.linspace(0, 10, 1000, endpoint=False)  # First 10 seconds (1000 samples at 100 Hz)

demo_keys = [
    "single_eeg_c3",
    "sim_avg_eeg",
    "sim_bipolar_eeg",
    "single_eog_l",
    "sim_avg_eog",
    "sim_bipolar_eog",
]

fig, axes = plt.subplots(len(demo_keys), 1, figsize=(12, 8), sharex=True)
for ax, key in zip(axes, demo_keys):
    x_uv, ch_names = apply_channel_config(sample_subj, shhs_configs[key], normalize=False)
    ax.plot(t_sec, x_uv[ep_idx, 0, :1000], linewidth=1.0)
    ax.set_ylabel(f"{ch_names[0]}\n(uV)", fontsize=9)
    ax.set_title(f"{key}: {shhs_configs[key]['description']}", fontsize=9.5, loc="left")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time within 30s N3 Epoch (seconds)")
fig.suptitle(f"Comparison of Location-Close & Mathematically Simulated Channels ({sample_subj['subject_id']}, Epoch #{ep_idx} [N3])", fontsize=12)
fig.tight_layout()
plt.show()

---
## Step 3: TinySleepNet Architecture (`use_context=False` & `use_context=True` Ready to Toggle)

Let's instantiate `TinySleepNet` in both **Model A (`use_context=False`, Context-Blind CNN)** and **Model B (`use_context=True`, Context-Aware CNN + LSTM)** modes and verify forward pass tensor shapes and parameter counts.

In [ ]:
for ctx_mode in (False, True):
    model = build_tinysleepnet_from_config(base_config, in_channels=1, use_context_override=ctx_mode)
    n_params_total = sum(p.numel() for p in model.parameters())
    n_params_cnn = sum(p.numel() for p in model.feature_extractor.parameters())
    n_params_head = sum(p.numel() for p in model.classifier.parameters())
    
    if not ctx_mode:
        dummy_in = torch.randn(4, 1, 3000)
    else:
        dummy_in = torch.randn(2, 20, 1, 3000)  # Batch=2, Seq_Len=20, C=1, Samples=3000
    dummy_out = model(dummy_in)
    print(
        f"TinySleepNet (use_context={str(ctx_mode):5s}) | "
        f"Input: {tuple(dummy_in.shape)} -> Output Logits: {tuple(dummy_out.shape)} | "
        f"feat_dim={model.feat_dim:,} | Params: Total={n_params_total:,} (CNN={n_params_cnn:,}, Head={n_params_head:,})"
    )

---
## Step 4: Subject-Independent 10-Fold Cross-Validation & Training (`src/training/trainer.py`)

First, we generate (or load) the strict **subject-independent 10-fold CV split** (`data/splits/shhs_10fold_splits.json`), where each fold uses **8 subjects (80%) for Training**, **1 subject (10%) for Validation**, and **1 subject (10%) for Test**.

### Automatic Saving & Instant Resume Across Sessions / Computers
- **Per-Fold Caching:** As soon as each fold finishes, its model checkpoint (`.pt`) and complete evaluation & hypnogram results (`results/<run_dir>/folds/<config>/fold_<XX>.json` and `results/<run_dir>/metrics_<config>.json`) are saved to disk.
- **Never Re-Train Completed Folds (`RESUME_FROM_SAVED = True`, `FORCE_RETRAIN = False`):**
  - Any fold already completed (such as Folds 0 and 1 from an earlier run) is loaded from disk in milliseconds and skipped.
  - When you reopen this notebook later or on a different computer (with the `results/` folder present), all 10 completed folds load instantly from disk without retraining!

In [ ]:
split_file = PROJECT_ROOT / base_config["paths"]["splits_dir"] / "shhs_10fold_splits.json"
splits = generate_subject_cv_splits(
    subject_ids=[r["subject_id"] for r in all_records],
    n_folds=10,
    seed=int(base_config["seed"]),
    save_path=split_file,
)

df_splits = pd.DataFrame([
    {
        "Fold": s["fold"],
        "Train_Subjects (80%)": ", ".join(s["train"]),
        "Val_Subject (10%)": ", ".join(s["val"]),
        "Test_Subject (10%)": ", ".join(s["test"]),
    }
    for s in splits
])
display(df_splits)

In [ ]:
# ============================================================================
# Experiment Control Panel (10-Fold Cross-Validation with Persistent Caching)
# ============================================================================
USE_CONTEXT = False         # False = Model A (Context-Blind CNN); True = Model B (Context-Aware LSTM)
MAX_EPOCHS = 5              # Set to 200 for full convergence, or 5-15 for testing on the 10 SHHS files
MAX_FOLDS = 10              # 10 = Full 10-fold CV across all 10 SHHS subjects
RESUME_FROM_SAVED = True    # True = Automatically load any completed folds/configs from disk without retraining
FORCE_RETRAIN = False       # Only set to True if you want to overwrite saved results and retrain from scratch

# Representative configs spanning all 3 categories (or set to `list(shhs_configs.keys())` for all 12)
SELECTED_CONFIG_NAMES = list(shhs_configs.keys())

results_dir = PROJECT_ROOT / base_config["paths"]["results_dir"] / ("notebook_ctx_true" if USE_CONTEXT else "notebook_ctx_false")
ablation_results = {}
ablation_summaries = {}

for cfg_name in SELECTED_CONFIG_NAMES:
    cfg_spec = shhs_configs[cfg_name]
    print(f"\n{'='*78}")
    print(f"Config: {cfg_name} ({cfg_spec['description']})")
    print(f"{'='*78}")
    res = run_cross_validation_for_config(
        all_records=all_records,
        splits=splits,
        channel_config=cfg_spec,
        base_config=base_config,
        config_name=cfg_name,
        use_context_override=USE_CONTEXT,
        max_epochs_override=MAX_EPOCHS,
        max_folds=MAX_FOLDS,
        results_dir=results_dir,
        resume=RESUME_FROM_SAVED,
        force_retrain=FORCE_RETRAIN,
        verbose=True,
    )
    ablation_results[cfg_name] = res
    ablation_summaries[cfg_name] = res["aggregated_metrics"]

# Save combined CSV summary table to disk
df_ablation_summary = metrics_summary_to_dataframe(ablation_summaries)
summary_csv_path = results_dir / "ablation_summary_10fold.csv"
df_ablation_summary.to_csv(summary_csv_path, index=False)
print(f"\nAll results saved to {results_dir} (Summary CSV: {summary_csv_path})")

---
## Step 5: Standardized Evaluation Metrics (Markov et al., 2025 & van der Aar et al., 2025)

> **Tip for New Sessions / Different Computers:**
> If you open this notebook on another computer or in a new session after the 10 folds have been saved in `results/`, you do **not** need to run Step 4 again—running the cell below automatically loads `ablation_results` and `ablation_summaries` directly from `results/`!

We now inspect and save:
1. **Overall & Per-Stage Epoch Classification Metrics** (Multiclass Cohen's $\kappa$, Multiclass MCC, Macro-F1, Weighted-F1, Micro vs. Macro averages, and stage-specific F1/SE/PPV/MCC).
2. **Row-Normalized Confusion Matrices** (`mean ± SD` across test folds).
3. **Whole-Night Clinical Sleep Architecture Parameters** (`TST`, `SE%`, `SOL`, `WASO`, stage durations) and **Bland–Altman** agreement plots + full-night hypnogram comparisons.

In [ ]:
# Automatically load from disk if opening the notebook in a fresh session without running Step 4
if "USE_CONTEXT" not in globals():
    USE_CONTEXT = False
if "SELECTED_CONFIG_NAMES" not in globals():
    SELECTED_CONFIG_NAMES = list(shhs_configs.keys())
if "results_dir" not in globals():
    results_dir = PROJECT_ROOT / base_config["paths"]["results_dir"] / ("notebook_ctx_true" if USE_CONTEXT else "notebook_ctx_false")

if not globals().get("ablation_results"):
    print(f"Loading saved results from {results_dir} ...")
    ablation_results, ablation_summaries = load_saved_ablation_results(results_dir, SELECTED_CONFIG_NAMES)

df_ablation_summary = metrics_summary_to_dataframe(ablation_summaries)
display(df_ablation_summary)

In [ ]:
# Plot & Save Normalized Confusion Matrices (mean +- SD across folds) for each evaluated configuration
active_cfgs = [c for c in SELECTED_CONFIG_NAMES if c in ablation_summaries]
n_cfgs = len(active_cfgs)
ncols = min(3, max(1, n_cfgs))
nrows = (n_cfgs + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 5.2 * nrows))
axes_flat = np.atleast_1d(axes).ravel()

for idx, cfg_name in enumerate(active_cfgs):
    summ = ablation_summaries[cfg_name]
    mean_cm = np.array(summ["mean_confusion_matrix_norm"])
    std_cm = np.array(summ["std_confusion_matrix_norm"])
    plot_normalized_confusion_matrix(
        mean_norm_cm=mean_cm,
        std_norm_cm=std_cm,
        title=f"{cfg_name} (κ={summ['pooled_metrics']['kappa']:.2f}, {ablation_results[cfg_name]['n_folds_run']} folds)",
        ax=axes_flat[idx],
    )

for idx in range(n_cfgs, len(axes_flat)):
    axes_flat[idx].axis("off")

fig.suptitle("Row-Normalized Confusion Matrices (Mean ± SD Across Test Folds)", fontsize=14)
fig.tight_layout()
cm_fig_path = results_dir / "figures" / "confusion_matrices_grid.png"
cm_fig_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(cm_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved confusion matrices figure to {cm_fig_path}")

In [ ]:
# Clinical Sleep Architecture Agreement Table & Hypnogram / Bland-Altman Visualization
target_cfg = active_cfgs[0]
clin_agree = ablation_results[target_cfg]["pooled_clinical_agreement"]
df_clinical = pd.DataFrame.from_dict(clin_agree, orient="index")
clin_csv_path = results_dir / f"clinical_agreement_{target_cfg}.csv"
df_clinical.to_csv(clin_csv_path)
print(f"Whole-Night Clinical Sleep Parameter Agreement for '{target_cfg}' (saved to {clin_csv_path}):")
display(df_clinical.round(2))

# Collect all test subject hypnograms for target_cfg
test_subject_records = [
    subj_rec
    for fold_out in ablation_results[target_cfg]["folds"]
    for subj_rec in fold_out["test_per_subject"]
]

if test_subject_records:
    first_test = test_subject_records[0]
    hyp_fig_path = results_dir / "figures" / f"hypnogram_{target_cfg}_{first_test['subject_id']}.png"
    plot_hypnogram_comparison(
        y_true=first_test["y_true"],
        y_pred=first_test["y_pred"],
        subject_id=first_test["subject_id"],
        config_name=target_cfg,
        save_path=hyp_fig_path,
    )
    plt.show()

    if len(test_subject_records) >= 2:
        ba_fig_path = results_dir / "figures" / f"bland_altman_{target_cfg}.png"
        plot_bland_altman_grid(
            subject_records=test_subject_records,
            parameters=["TST_min", "SE_pct", "SOL_min", "WASO_min"],
            title=f"Bland-Altman Agreement — {target_cfg} (N={len(test_subject_records)} Test Nights)",
            save_path=ba_fig_path,
        )
        plt.show()